# 08. 딥러닝 - 회귀 (Keras)

> 분류(08_딥러닝_분류_이진.ipynb / 08_딥러닝_분류_다중.ipynb)와 출력층 설계·손실함수가 완전히 달라 별도 파일로 분리했습니다. 05번 노트북과 같은 Titanic Fare 예측 문제를 이번에는 딥러닝(Keras Sequential)으로 풀어봅니다.

### 이 노트북에서 배우는 것
- Sequential 모델 설계 5단계
- 회귀용 출력층 설계(노드 1개, activation 없음)
- loss='mse' 등 회귀용 컴파일 옵션
- loss curve와 예측 vs 실제 산점도로 성능 확인

### 사용 데이터
- `data/train.csv` (Titanic) - Pclass/Age/SibSp/Parch/Fare로 Fare 예측

## 목차
1. 딥러닝 개요와 AICE 시험 범위
2. 회귀용 Sequential 모델 설계
3. 모델 컴파일
4. 모델 학습
5. 성능 시각화
6. 종합 실전 문제

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

tf.random.set_seed(100)  # 재현성을 위해 시드 고정

df = pd.read_csv('../data/train.csv')
df['Age'] = df['Age'].fillna(df['Age'].median())

X = df[['Pclass', 'Age', 'SibSp', 'Parch']]
y = df['Fare']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
X_train.shape

## 1. 딥러닝 개요와 AICE 시험 범위

### 📖 이론 설명
AICE Associate의 딥러닝 문제는 역전파의 수학적 유도 과정을 묻지 않습니다. 대신 **'이 문제 유형(회귀/이진분류/다중분류)에 맞게 출력층과 손실함수를 올바르게 설계할 수 있는가'**를 봅니다. Keras의 `Sequential` API로 층을 쌓고, `compile`로 학습 방식을 정하고, `fit`으로 학습시키는 흐름은 어떤 문제든 동일합니다 - **바뀌는 것은 출력층의 노드 수·activation과 loss뿐**입니다. 이 노트북(회귀)에서는 그 중 가장 단순한 형태를 다룹니다.

## 2. 회귀용 Sequential 모델 설계

### 📖 이론 설명
회귀는 '숫자 하나'를 예측하므로 출력층 노드는 **1개**이고, 그 값을 0~1이나 특정 범위로 제한할 이유가 없으므로 **activation을 지정하지 않습니다**(항등함수, 즉 그대로 출력). 중간(hidden) 레이어는 `relu`를 관용적으로 사용합니다.

### 🧩 핵심 개념 정리
| 레이어 | 노드 수 | activation |
|---|---|---|
| 입력층 | 특성 개수(input_shape) | - |
| 은닉층 | 자유롭게 설정 | 보통 'relu' |
| 출력층(회귀) | 1 | 없음(None) |

### 💻 예제 코드

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

model = Sequential()
model.add(Dense(16, activation='relu', input_shape=(X_train.shape[1],)))
model.add(Dense(8, activation='relu'))
model.add(Dense(1))  # 회귀: activation 없음
model.summary()

### ✍️ TODO: 실전 문제

**문제 1.** 입력층 8노드, 은닉층 4노드(둘 다 relu), 출력층 1노드(활성화 없음)로 구성된 `model2`를 만들고 `summary()`를 확인하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
model2 = Sequential()  # 레이어를 순서대로 쌓아올리는 가장 기본적인 Keras 모델 구조
model2.add(Dense(8, activation='relu', input_shape=(X_train.shape[1],)))  # input_shape은 첫 레이어에만 지정, 이후 레이어는 이전 출력 크기를 자동으로 이어받음
model2.add(Dense(4, activation='relu'))
model2.add(Dense(1))  # 회귀 출력층은 숫자 하나를 그대로 내야 하므로 activation을 지정하지 않음(항등함수)
model2.summary()
```

</details>

## 3. 모델 컴파일

### 📖 이론 설명
회귀 문제는 예측값과 실제값의 '차이(오차)'를 줄이는 것이 목표이므로 손실함수로 **`mse`(평균제곱오차)**를 씁니다. `metrics=['mae']`를 추가하면 학습 중 MAE도 함께 확인할 수 있습니다. 옵티마이저는 보통 `Adam`을 기본으로 사용합니다.

### 🧩 핵심 개념 정리
| 옵션 | 회귀에서의 값 |
|---|---|
| loss | 'mse' 또는 'mean_squared_error' |
| metrics | ['mae'] |
| optimizer | 'adam' (또는 Adam(learning_rate=...)) |

### 💻 예제 코드

In [ ]:
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

### ✍️ TODO: 실전 문제

**문제 1.** `model2`를 `optimizer='adam'`, `loss='mse'`, `metrics=['mae']`로 컴파일하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
model2.compile(optimizer='adam', loss='mse', metrics=['mae'])  # 회귀는 오차를 줄이는 것이 목표이므로 loss='mse', 학습 중 척도를 보기 위해 metrics=['mae']를 추가
```

</details>

## 4. 모델 학습

### 📖 이론 설명
`EarlyStopping`은 검증 손실(`val_loss`)이 `patience` epoch 동안 개선되지 않으면 학습을 자동으로 멈춰, 불필요한 학습(과적합)을 막아줍니다. `restore_best_weights=True`를 주면 가장 좋았던 시점의 가중치로 되돌립니다.

### 🧩 핵심 개념 정리
| 콜백/옵션 | 설명 |
|---|---|
| EarlyStopping(monitor='val_loss', patience=n) | n epoch 개선 없으면 조기종료 |
| validation_data=(X_test, y_test) | 이미 분리해둔 검증셋을 그대로 사용 |
| validation_split=0.2 | 검증셋을 따로 안 만들었을 때, 학습 데이터의 뒤쪽 20%를 매 epoch 검증용으로 자동 분리 |
| epochs / batch_size | 전체 학습 반복 수 / 한 번에 학습할 데이터 개수 |

### 💻 예제 코드

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
history = model.fit(X_train, y_train, epochs=100, batch_size=32,
                     validation_data=(X_test, y_test), callbacks=[es], verbose=0)
print('실제 학습된 epoch 수:', len(history.history['loss']))

### ✍️ TODO: 실전 문제

**문제 1.** `model2`를 `epochs=50`, `batch_size=16`, `validation_data=(X_test, y_test)`로 학습시키세요. (verbose=0)

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
history2 = model2.fit(X_train, y_train, epochs=50, batch_size=16, validation_data=(X_test, y_test), verbose=0)  # validation_data를 주면 매 epoch마다 검증 성능도 함께 기록됨
print(len(history2.history['loss']))  # history.history는 epoch별 loss가 담긴 딕셔너리라 길이로 실제 학습된 epoch 수를 확인 가능
```

</details>

**문제 2.** `model2`를 이번엔 `validation_data` 대신 `validation_split=0.2`로 `epochs=50`, `batch_size=16`으로 학습시키세요. (verbose=0)

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
history3 = model2.fit(X_train, y_train, epochs=50, batch_size=16, validation_split=0.2, verbose=0)  # 별도 검증셋(X_test) 없이 X_train의 뒤쪽 20%를 학습에서 제외하고 검증용으로 자동 분리
print(len(history3.history['loss']))
```

</details>

## 5. 성능 시각화

### 📖 이론 설명
`history.history`에는 epoch마다의 loss/val_loss(그리고 지정한 metrics)가 리스트로 저장되어 있습니다. 이를 그려보면 학습이 잘 되고 있는지(loss가 꾸준히 감소하는지), 과적합이 일어나는지(train loss는 내려가는데 val loss는 다시 올라가는지)를 확인할 수 있습니다.

### 💻 예제 코드

In [ ]:
plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='val loss')
plt.xlabel('Epoch'); plt.ylabel('MSE Loss'); plt.title('Training Loss Curve')
plt.legend(); plt.show()

pred = model.predict(X_test).flatten()
plt.scatter(y_test, pred, alpha=0.5)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
plt.xlabel('Actual Fare'); plt.ylabel('Predicted Fare'); plt.title('DL Regression: Actual vs Predicted')
plt.show()

### ✍️ TODO: 실전 문제

**문제 1.** `model2`의 `mae`, `val_mae` 곡선을 그리세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
plt.plot(history2.history['mae'], label='train mae')
plt.plot(history2.history['val_mae'], label='val mae')  # train과 val 곡선이 벌어지기 시작하면 과적합의 신호
plt.xlabel('Epoch'); plt.ylabel('MAE'); plt.legend(); plt.show()
```

</details>

## 6. 종합 실전 문제

**딥러닝 회귀 전체 흐름을 이어서 풀어보는 미니 모의고사입니다.**

**문제 1.** 입력층 32노드(relu), 은닉층 16노드(relu), 출력층 1노드(활성화 없음)로 `model3`을 만들고 `optimizer='adam', loss='mse'`로 컴파일하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
model3 = Sequential()
model3.add(Dense(32, activation='relu', input_shape=(X_train.shape[1],)))
model3.add(Dense(16, activation='relu'))
model3.add(Dense(1))  # 출력 노드 1개, activation 없음 = 회귀 모델의 표준 형태
model3.compile(optimizer='adam', loss='mse', metrics=['mae'])
model3.summary()
```

</details>

**문제 2.** `model3`을 `EarlyStopping(monitor='val_loss', patience=5)`와 함께 `epochs=100`으로 학습시키고, 최종 테스트 MSE를 구하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
es3 = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)  # val_loss가 5 epoch 동안 개선되지 않으면 학습을 멈추고, 가장 좋았던 시점의 가중치로 복원
history3 = model3.fit(X_train, y_train, epochs=100, batch_size=32, validation_data=(X_test, y_test), callbacks=[es3], verbose=0)
test_loss, test_mae = model3.evaluate(X_test, y_test, verbose=0)  # evaluate()는 학습에 쓰이지 않은 데이터로 최종 성능만 확인
print(test_loss, test_mae)
```

</details>